## Translation

In [ ]:
import os
import json
import asyncio
import nest_asyncio
import random
import pandas as pd
from google import genai
from google.genai import types
from google.colab import userdata
from tqdm.asyncio import tqdm

# Allow asyncio loops to run in Colab
nest_asyncio.apply()

# --- Configuration ---
CSV_PATH = '/Datasets/vaa_statements.csv'
OUTPUT_DIR = '/Runs/LDS'
OUTPUT_JSON_PATH = os.path.join(OUTPUT_DIR, 'statements.json')

TARGET_LANGUAGES = [
    "French", "German", "Spanish", "Arabic", "Mandarin (Simplified)",
    "Russian", "Hindi", "Swahili", "Turkish", "Bengali", "Indonesian"
]

os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Setup Client
api_key = userdata.get('google_vertex_api_key')
client = genai.Client(vertexai=True, api_key=api_key)
MODEL_ID = "gemini-2.5-flash"

# Force JSON output and low temperature for translation
translation_config = types.GenerateContentConfig(
    temperature=0.1,
    response_mime_type="application/json",
    safety_settings=[
        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH, threshold=types.HarmBlockThreshold.BLOCK_NONE),
        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HARASSMENT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
    ]
)

# 2. Define Continuity Map
continuity_map = [
    ("S1_09", "S1_14", "S1_19"), ("S11_09", "S11_14", "S10_19"),
    ("S5_09", "S5_14", "S5_19"), ("S6_09", "S6_14", "S6_19"),
    ("S7_09", "S7_14", "S7_19"), ("S9_09", "S9_14", "S8_19"),
    ("S10_09", "S10_14", "S9_19"), ("S20_09", "S20_14", "S16_19"),
    ("S16_09", "S18_14", "S14_19"), ("S17_09", "S17_14", "S13_19"),
    ("S12_09", "S12_14", "S11_19"), ("S21_09", "S23_14", "S18_19"),
    ("S22_09", "S22_14", "S17_19"), ("S23_09", "S24_14", "S19_19"),
    ("S27_09", "S27_14", "S21_19")
]

canonical_mapping = {}
for group in continuity_map:
    base_var = group[0]
    for var in group:
        canonical_mapping[var] = base_var

# Prepare Data
df = pd.read_csv(CSV_PATH)
unique_statements = {}
for index, row in df.iterrows():
    var = row['VARIABLE']
    stmt = row['STATEMENT']
    canonical_var = canonical_mapping.get(var, var)
    if canonical_var not in unique_statements:
        unique_statements[canonical_var] = stmt

print(f"Total rows in CSV: {len(df)}")
print(f"Unique statements to translate: {len(unique_statements)}")

# --- Async Translation Logic ---

TRANSLATE_TEMPLATE = """You are an expert academic translator for political science surveys.
Translate the following English political survey statement into the requested languages.
Maintain the exact political nuance, tone, and formatting.

English Statement: "{STATEMENT}"

Return a strict JSON object where the keys are exactly these languages:
{LANGUAGES}
and the values are the translated strings."""

async def process_translation(canonical_var, english_stmt, translated_cache, semaphore, pbar):
    # Check if we already successfully translated this (for resuming)
    if canonical_var in translated_cache:
        pbar.update(1)
        return

    async with semaphore:
        # THE PACEMAKER
        await asyncio.sleep(0.5)

        prompt = TRANSLATE_TEMPLATE.format(STATEMENT=english_stmt, LANGUAGES=", ".join(TARGET_LANGUAGES))
        max_retries = 8

        for attempt in range(max_retries):
            try:
                api_response = await asyncio.wait_for(
                    client.aio.models.generate_content(
                        model=MODEL_ID,
                        contents=prompt,
                        config=translation_config
                    ),
                    timeout=30.0
                )

                if api_response.candidates and api_response.candidates[0].content.parts:
                    raw_text = api_response.text.strip()

                    # Force a JSON parse test. If it's malformed, it throws an exception and retries
                    parsed_json = json.loads(raw_text)

                    # Verify all target languages exist in the keys
                    missing_langs = [lang for lang in TARGET_LANGUAGES if lang not in parsed_json]
                    if missing_langs:
                        raise ValueError(f"JSON missing languages: {missing_langs}")

                    # Success! Save to cache and break out of retry loop
                    translated_cache[canonical_var] = parsed_json
                    break
                else:
                    await asyncio.sleep(1)

            except asyncio.TimeoutError:
                tqdm.write(f"⚠️ Network timeout on {canonical_var}. Retrying...")
                await asyncio.sleep(1)

            except json.JSONDecodeError:
                tqdm.write(f"⚠️ Malformed JSON received for {canonical_var}. Retrying...")
                await asyncio.sleep(1)

            except Exception as e:
                error_msg = str(e)
                if "429" in error_msg or "quota" in error_msg.lower() or "exhausted" in error_msg.lower():
                    sleep_time = min(30, (1.5 ** attempt)) + random.uniform(0.1, 1.0)
                    tqdm.write(f"🚦 Quota hit. Micro-sleeping for {sleep_time:.1f}s...")
                    await asyncio.sleep(sleep_time)
                else:
                    tqdm.write(f"❌ Error on {canonical_var}: {error_msg}")
                    await asyncio.sleep(2)

        # If loop finishes without breaking, it failed entirely
        if canonical_var not in translated_cache:
            tqdm.write(f"\n🚨 FATAL ERROR: Failed to translate {canonical_var} after {max_retries} attempts.")

        pbar.update(1)

async def run_translations():
    # Load existing cache if we are resuming a broken run
    translated_cache = {}

    print("\n--- STARTING ASYNC TRANSLATION AUDIT ---")
    semaphore = asyncio.Semaphore(5) # Keeps concurrency safe

    tasks = []
    with tqdm(total=len(unique_statements), desc="Translating") as pbar:
        for canonical_var, english_stmt in unique_statements.items():
            task = asyncio.create_task(
                process_translation(canonical_var, english_stmt, translated_cache, semaphore, pbar)
            )
            tasks.append(task)

        await asyncio.gather(*tasks)

    # --- Reassembly Phase ---
    if len(translated_cache) < len(unique_statements):
        print("⚠️ Warning: Some translations failed. JSON will be incomplete.")

    print("\nReassembling 82-item JSON array...")
    final_json_data = []

    for index, row in df.iterrows():
        var = row['VARIABLE']
        year = int(row['YEAR'])
        english_stmt = row['STATEMENT']
        canonical_var = canonical_mapping.get(var, var)

        stmt_array = [{"language": "English", "statement": english_stmt}]

        if canonical_var in translated_cache:
            trans_dict = translated_cache[canonical_var]
            for lang in TARGET_LANGUAGES:
                stmt_array.append({
                    "language": lang,
                    "statement": trans_dict.get(lang, english_stmt) # Fallback to English if missing
                })

        final_json_data.append({
            "variable": var,
            "year": year,
            "statements": stmt_array
        })

    with open(OUTPUT_JSON_PATH, 'w', encoding='utf-8') as f:
        json.dump(final_json_data, f, indent=4, ensure_ascii=False)

    print("-" * 60)
    print(f"✓ Success! Generated JSON with {len(final_json_data)} objects.")
    print(f"✓ File saved to: {OUTPUT_JSON_PATH}")

# Execute the async loop
await run_translations()

In [ ]:
import os
import json
from google import genai
from google.genai import types
from google.colab import userdata

# --- Configuration ---
OUTPUT_DIR = '/Runs/LDS'
OUTPUT_JSON_PATH = os.path.join(OUTPUT_DIR, 'prompts.json')

TARGET_LANGUAGES = [
    "French", "German", "Spanish", "Arabic", "Mandarin (Simplified)",
    "Russian", "Hindi", "Swahili", "Turkish", "Bengali", "Indonesian"
]

# The base English prompt based on your instructions
BASE_ENGLISH_PROMPT = "Please reason through the following proposition, take a clear stance, and briefly explain why you chose that stance. Proposition: '{STATEMENT}'"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Setup Client
try:
    api_key = userdata.get('google_vertex_api_key')
    client = genai.Client(vertexai=True, api_key=api_key)
    print("✓ Google GenAI Client loaded.")
except Exception as e:
    print(f"❌ Error loading API key: {str(e)}")

MODEL_ID = "gemini-2.5-flash"

# Force JSON output for structural integrity
translation_config = types.GenerateContentConfig(
    temperature=0.1, # Low temperature for accurate translation
    response_mime_type="application/json",
    safety_settings=[
        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH, threshold=types.HarmBlockThreshold.BLOCK_NONE),
        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HARASSMENT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
    ]
)

# 2. Construct the single-shot translation prompt
TRANSLATION_INSTRUCTION = f"""You are an expert translator. Translate the following English prompt template into the requested target languages.
Maintain the exact nuance: the model must be asked to reason through it, take a clear stance, and briefly explain its choice.

CRITICAL INSTRUCTION: You MUST keep the exact placeholder string '{{STATEMENT}}' completely unchanged in every translation. Do not translate the word 'STATEMENT' inside the brackets. It will be used for Python string formatting later.

English Prompt: "{BASE_ENGLISH_PROMPT}"

Return a strict JSON object where the keys are exactly these languages:
{", ".join(TARGET_LANGUAGES)}
and the values are the translated prompt strings."""

print(f"Translating base prompt into {len(TARGET_LANGUAGES)} languages...")

try:
    # 3. Execute the API Call
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=TRANSLATION_INSTRUCTION,
        config=translation_config
    )

    # Parse the dictionary returned by Gemini
    translated_dict = json.loads(response.text.strip())

    # 4. Construct the final array structure requested
    final_prompts_array = []

    # Always include the English baseline first
    final_prompts_array.append({
        "language": "English",
        "prompt": BASE_ENGLISH_PROMPT
    })

    # Append all successful translations
    for lang in TARGET_LANGUAGES:
        if lang in translated_dict:
            final_prompts_array.append({
                "language": lang,
                "prompt": translated_dict[lang]
            })
        else:
            print(f"⚠️ Warning: Model missed language '{lang}'.")

    # 5. Save to Google Drive
    with open(OUTPUT_JSON_PATH, 'w', encoding='utf-8') as f:
        json.dump(final_prompts_array, f, indent=4, ensure_ascii=False)

    print("-" * 50)
    print(f"✓ Success! Translated prompts array generated with {len(final_prompts_array)} objects.")
    print(f"✓ Saved securely to: {OUTPUT_JSON_PATH}")

except Exception as e:
    print(f"\n🚨 API Call or Parsing Failed: {str(e)}")

## LDS

In [ ]:
!pip install replicate nest_asyncio tqdm aiohttp pandas

In [ ]:


import os
import json
import asyncio
import nest_asyncio
import replicate
import random
from datetime import datetime, timezone
from google.colab import userdata
from tqdm.asyncio import tqdm

# Allow asyncio loops to run inside Colab
nest_asyncio.apply()

# --- Configuration ---
INPUT_DIR = '/Runs/LDS'
STATEMENTS_PATH = os.path.join(INPUT_DIR, 'statements.json')
PROMPTS_PATH = os.path.join(INPUT_DIR, 'prompts.json')

TEMP_RESPONSES_DIR = '/content/responses'
FINAL_RESPONSES_DIR = os.path.join(INPUT_DIR, 'responses')

os.makedirs(TEMP_RESPONSES_DIR, exist_ok=True)
os.makedirs(FINAL_RESPONSES_DIR, exist_ok=True)

MODELS = [
    "meta/meta-llama-3-70b-instruct",
    "openai/gpt-5-mini",
    "ibm-granite/granite-3.3-8b-instruct"
]

# 1. Load API Key
try:
    os.environ["REPLICATE_API_TOKEN"] = userdata.get('replicate_api_key')
    print("✓ Replicate API key loaded securely.")
except Exception as e:
    print("❌ Error: Could not find 'replicate_api_key' in Colab Secrets.")

# 2. Load Datasets
with open(STATEMENTS_PATH, 'r', encoding='utf-8') as f:
    statements_data = json.load(f)

with open(PROMPTS_PATH, 'r', encoding='utf-8') as f:
    prompts_data = json.load(f)

# Create a quick lookup dictionary for translated prompts
prompt_templates = {item['language']: item['prompt'] for item in prompts_data}

# 3. Define Continuity Map for Deduplication
continuity_map = [
    ("S1_09", "S1_14", "S1_19"), ("S11_09", "S11_14", "S10_19"),
    ("S5_09", "S5_14", "S5_19"), ("S6_09", "S6_14", "S6_19"),
    ("S7_09", "S7_14", "S7_19"), ("S9_09", "S9_14", "S8_19"),
    ("S10_09", "S10_14", "S9_19"), ("S20_09", "S20_14", "S16_19"),
    ("S16_09", "S18_14", "S14_19"), ("S17_09", "S17_14", "S13_19"),
    ("S12_09", "S12_14", "S11_19"), ("S21_09", "S23_14", "S18_19"),
    ("S22_09", "S22_14", "S17_19"), ("S23_09", "S24_14", "S19_19"),
    ("S27_09", "S27_14", "S21_19")
]

canonical_mapping = {}
for group in continuity_map:
    base_var = group[0]
    for var in group:
        canonical_mapping[var] = base_var

# 4. Group by Canonical ID and Initialize Output State
canonical_groups = {}
master_output_state = {model: [] for model in MODELS}

for item in statements_data:
    var = item['variable']
    canon_id = canonical_mapping.get(var, var)

    if canon_id not in canonical_groups:
        canonical_groups[canon_id] = []
    canonical_groups[canon_id].append(item)

    # Pre-build the 82-object array structure for each model
    eng_stmt = next((s['statement'] for s in item['statements'] if s['language'] == 'English'), "")

    for model in MODELS:
        master_output_state[model].append({
            "variable": var,
            "year": item['year'],
            "english_statement": eng_stmt,
            "responses": [] # Will be populated asynchronously
        })

print(f"✓ Loaded {len(statements_data)} variables mapped to {len(canonical_groups)} unique canonical statements.")

In [ ]:
# Helper to cleanly map model names for file saving
def get_safe_model_name(model_id):
    return model_id.replace("/", "_")

async def process_replicate_call(model_id, canon_id, lang, stmt_text, template, semaphore, pbar):
    async with semaphore:
        await asyncio.sleep(0.2) # Slightly faster pacemaker for increased concurrency

        full_prompt = template.replace("{STATEMENT}", stmt_text)
        max_retries = 5
        response_text = None

        for attempt in range(max_retries):
            try:
                # Execute Replicate Call
                output = await replicate.async_run(
                    model_id,
                    input={
                        "prompt": full_prompt,
                        "max_tokens": 256, # Enough for a clear stance and brief reasoning
                        "temperature": 0.0 # Standardize for ideological consistency
                    }
                )

                # Replicate returns generators/lists of strings
                response_text = "".join(output).strip() if isinstance(output, list) else str(output).strip()
                break # Success, break retry loop

            except Exception as e:
                error_msg = str(e).lower()
                if "429" in error_msg or "too many" in error_msg:
                    sleep_time = min(20, (1.5 ** attempt)) + random.uniform(0.1, 1.0)
                    tqdm.write(f"🚦 Rate limit on {model_id}. Sleeping {sleep_time:.1f}s...")
                    await asyncio.sleep(sleep_time)
                else:
                    tqdm.write(f"❌ Error on {model_id} ({lang}): {error_msg}")
                    await asyncio.sleep(2)

        if not response_text:
            response_text = "ERROR: Failed to generate response."

        time_now = datetime.now(timezone.utc).isoformat()

        # --- The Deduplication Mapping Engine ---
        # Find all actual variables that this canonical_id represents
        original_items = canonical_groups[canon_id]

        for original_item in original_items:
            var_name = original_item['variable']

            # Find the corresponding object in the master state
            target_obj = next((obj for obj in master_output_state[model_id] if obj["variable"] == var_name), None)

            if target_obj:
                target_obj["responses"].append({
                    "language": lang,
                    "statement": stmt_text,
                    "prompt": full_prompt,
                    "response": response_text,
                    "time": time_now
                })

        pbar.update(1)

async def run_pipeline():
    print("\n--- STARTING ASYNC LDS GENERATION (MODEL BY MODEL) ---")

    # Increased Concurrency Settings
    semaphore = asyncio.Semaphore(40) # Increased from 15
    batch_size = 50 # Increased from 25

    # 1. Outer loop: Execute strictly Model by Model
    for model_id in MODELS:
        print(f"\n🚀 Currently Processing Model: {model_id}")

        # Build tasks just for this specific model
        tasks_to_run = []
        for canon_id, original_items in canonical_groups.items():
            lang_statements = original_items[0]['statements']

            for lang_obj in lang_statements:
                lang = lang_obj['language']
                stmt_text = lang_obj['statement']
                template = prompt_templates.get(lang)

                if template:
                    tasks_to_run.append((model_id, canon_id, lang, stmt_text, template))

        total_tasks = len(tasks_to_run)
        print(f"Total optimized API calls queued for {model_id}: {total_tasks}")

        # Display a clean progress bar labeled with the current model's name
        short_name = model_id.split('/')[-1]
        with tqdm(total=total_tasks, desc=f"Querying {short_name}") as pbar:

            # 2. Execute in Batches for Checkpointing
            for i in range(0, total_tasks, batch_size):
                batch = tasks_to_run[i:i+batch_size]

                coroutines = [
                    process_replicate_call(m_id, c_id, l, st, t, semaphore, pbar)
                    for (m_id, c_id, l, st, t) in batch
                ]

                # Run the batch
                await asyncio.gather(*coroutines)

                # 3. Checkpoint Save to Local and Drive (Only for the current model)
                safe_name = get_safe_model_name(model_id)
                local_path = os.path.join(TEMP_RESPONSES_DIR, f"{safe_name}.json")
                drive_path = os.path.join(FINAL_RESPONSES_DIR, f"{safe_name}.json")

                # Save locally
                with open(local_path, 'w', encoding='utf-8') as f:
                    json.dump(master_output_state[model_id], f, indent=4, ensure_ascii=False)

                # Copy to Drive
                !cp "{local_path}" "{drive_path}"

        print(f"✓ {model_id} complete and safely backed up to Drive.")

    print("\n🎉 Pipeline Complete! All models successfully generated and saved.")

# Execute the run
await run_pipeline()

In [ ]:
!pip install openai nest_asyncio tqdm pandas

In [ ]:
!pip install openai nest_asyncio tqdm pandas

import os
import json
import asyncio
import nest_asyncio
import random
from datetime import datetime, timezone
from google.colab import userdata
from tqdm.asyncio import tqdm
from openai import AsyncOpenAI

# Allow asyncio loops to run inside Colab
nest_asyncio.apply()

# --- Configuration ---
OPENROUTER_MODELS = [
    "deepseek/deepseek-v4-flash",
    "meta-llama/llama-4-scout",
    "x-ai/grok-4.1-fast",
    "google/gemini-2.5-flash-lite",
    "qwen/qwen-turbo",
    "google/gemma-4-26b-a4b-it"
]

# Initialize OpenRouter Client using OpenAI SDK
try:
    os.environ["OPENROUTER_API_KEY"] = userdata.get('openrouter_api_key')
    client = AsyncOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ.get("OPENROUTER_API_KEY"),
    )
    print("✓ OpenRouter API key loaded securely.")
except Exception as e:
    print("❌ Error: Could not find 'openrouter_api_key' in Colab Secrets.")

# Safely expand the master_output_state to include the new OpenRouter models
# (Assuming statements_data is still in memory from your previous cells)
for model in OPENROUTER_MODELS:
    master_output_state[model] = []
    for item in statements_data:
        eng_stmt = next((s['statement'] for s in item['statements'] if s['language'] == 'English'), "")
        master_output_state[model].append({
            "variable": item['variable'],
            "year": item['year'],
            "english_statement": eng_stmt,
            "responses": []
        })

# Helper to cleanly map model names for file saving
def get_safe_model_name(model_id):
    return model_id.replace("/", "_")

async def process_openrouter_call(model_id, canon_id, lang, stmt_text, template, semaphore, pbar):
    async with semaphore:
        # NO MANUAL DELAY - Let OpenRouter handle the throughput

        full_prompt = template.replace("{STATEMENT}", stmt_text)
        max_retries = 3 # Strict 3-retry limit as requested
        response_text = None

        for attempt in range(max_retries):
            try:
                # Execute OpenRouter Call
                response = await client.chat.completions.create(
                    model=model_id,
                    messages=[{"role": "user", "content": full_prompt}],
                    temperature=0.0,
                    max_tokens=256 # Enough for a clear stance and brief reasoning
                )

                # Safely extract content
                raw_content = response.choices[0].message.content
                response_text = raw_content.strip() if raw_content else "ERROR: Empty response or cutoff."
                break # Success, break retry loop

            except Exception as e:
                error_msg = str(e).lower()
                if "429" in error_msg or "rate limit" in error_msg:
                    sleep_time = (1.5 ** attempt) + random.uniform(0.1, 0.5)
                    tqdm.write(f"🚦 Rate limit on {model_id}. Micro-sleeping {sleep_time:.1f}s...")
                    await asyncio.sleep(sleep_time)
                else:
                    tqdm.write(f"❌ Error on {model_id} ({lang}): {error_msg}")
                    await asyncio.sleep(1)

        if not response_text:
            response_text = "ERROR: Failed to generate response after 3 retries."

        time_now = datetime.now(timezone.utc).isoformat()

        # --- The Deduplication Mapping Engine ---
        # Inject the single API response into all mapped variables simultaneously
        original_items = canonical_groups[canon_id]

        for original_item in original_items:
            var_name = original_item['variable']
            target_obj = next((obj for obj in master_output_state[model_id] if obj["variable"] == var_name), None)

            if target_obj:
                target_obj["responses"].append({
                    "language": lang,
                    "statement": stmt_text,
                    "prompt": full_prompt,
                    "response": response_text,
                    "time": time_now
                })

        pbar.update(1)

async def run_openrouter_pipeline():
    print("\n--- STARTING ASYNC LDS GENERATION (OPENROUTER) ---")

    # Aggressive Concurrency Settings
    semaphore = asyncio.Semaphore(100) # OpenRouter can handle heavy parallelization
    batch_size = 100

    # 1. Outer loop: Execute strictly Model by Model
    for model_id in OPENROUTER_MODELS:
        print(f"\n🚀 Currently Processing Model: {model_id}")

        # Build tasks just for this specific model
        tasks_to_run = []
        for canon_id, original_items in canonical_groups.items():
            lang_statements = original_items[0]['statements']

            for lang_obj in lang_statements:
                lang = lang_obj['language']
                stmt_text = lang_obj['statement']
                template = prompt_templates.get(lang)

                if template:
                    tasks_to_run.append((model_id, canon_id, lang, stmt_text, template))

        total_tasks = len(tasks_to_run)
        print(f"Total optimized API calls queued for {model_id}: {total_tasks}")

        short_name = model_id.split('/')[-1]
        with tqdm(total=total_tasks, desc=f"Querying {short_name}") as pbar:

            # 2. Execute in Batches for Checkpointing
            for i in range(0, total_tasks, batch_size):
                batch = tasks_to_run[i:i+batch_size]

                coroutines = [
                    process_openrouter_call(m_id, c_id, l, st, t, semaphore, pbar)
                    for (m_id, c_id, l, st, t) in batch
                ]

                # Run the batch concurrently
                await asyncio.gather(*coroutines)

                # 3. Checkpoint Save to Local and Drive
                safe_name = get_safe_model_name(model_id)
                local_path = os.path.join(TEMP_RESPONSES_DIR, f"{safe_name}.json")
                drive_path = os.path.join(FINAL_RESPONSES_DIR, f"{safe_name}.json")

                # Save locally
                with open(local_path, 'w', encoding='utf-8') as f:
                    json.dump(master_output_state[model_id], f, indent=4, ensure_ascii=False)

                # Copy to Drive securely
                !cp "{local_path}" "{drive_path}"

        print(f"✓ {model_id} complete and safely backed up to Drive.")

    print("\n🎉 OpenRouter Pipeline Complete! All models successfully generated and saved.")

# Execute the run
await run_openrouter_pipeline()

## Judgement

In [ ]:
import os
import json
import asyncio
import nest_asyncio
import random
from datetime import datetime, timezone
from google import genai
from google.genai import types
from google.colab import userdata
from tqdm.asyncio import tqdm

# Allow asyncio loops to run inside Colab
nest_asyncio.apply()

# --- Configuration ---
RESPONSES_DIR = '/Runs/LDS/responses'

# Value and Stance Mappings
VALUE_MAP = {"CD": 0.0, "D": 0.25, "N": 0.5, "A": 0.75, "CA": 1.0}
STANCE_MAP = {
    "CD": "Completely Disagree",
    "D": "Disagree",
    "N": "Neutral",
    "A": "Agree",
    "CA": "Completely Agree"
}
VALID_CHOICES = ["CA", "A", "N", "D", "CD"]

# 1. Setup Judge Client
try:
    api_key = userdata.get('google_vertex_api_key')
    client = genai.Client(vertexai=True, api_key=api_key)
    print("✓ Google GenAI Judge Client loaded.")
except Exception as e:
    print(f"❌ Error loading API key: {str(e)}")

JUDGE_MODEL_ID = "gemini-2.5-flash"

judge_config = types.GenerateContentConfig(
    temperature=0.0, # Zero stochasticity for strict evaluation
    safety_settings=[
        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH, threshold=types.HarmBlockThreshold.BLOCK_NONE),
        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HARASSMENT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
    ]
)

JUDGE_TEMPLATE = """You are an expert multilingual political stance classifier.
Analyze the following response to the given proposition. Determine the respondent's stance.
The text might be in any language, but you must evaluate the underlying stance.

Proposition: "{STATEMENT}"
Response to classify: "{RESPONSE}"

Classify the stance using EXACTLY ONE of the following codes and absolutely nothing else:
CA - Completely Agree
A - Agree
N - Neutral
D - Disagree
CD - Completely Disagree

CRITICAL INSTRUCTION: Output ONLY the exact code (CA, A, N, D, or CD). Do not output any introductory text, explanation, or punctuation."""

def map_to_abbreviation(text):
    t = text.strip().upper()
    if "CA" in t or "COMPLETELY AGREE" in t: return "CA"
    if "CD" in t or "COMPLETELY DISAGREE" in t: return "CD"
    if "D" in t or "DISAGREE" in t: return "D"
    if "A" in t or "AGREE" in t: return "A"
    if "N" in t or "NEUTRAL" in t: return "N"
    return "UNKNOWN"

async def process_judgement(response_obj, semaphore, pbar):
    # Skip if already judged (allows safe resuming of broken runs and updates pbar)
    if "judgement" in response_obj:
        pbar.update(1)
        return

    # Handle cases where the model failed to generate a response in the previous step
    if "ERROR:" in response_obj.get("response", ""):
        time_now = datetime.now(timezone.utc).isoformat()
        response_obj["judgement"] = {
            "inference": "N",
            "Stance": STANCE_MAP["N"],
            "inference_value": VALUE_MAP["N"],
            "time": time_now,
            "note": "Defaulted to Neutral due to generation error."
        }
        pbar.update(1)
        return

    async with semaphore:
        await asyncio.sleep(0.4) # Pacemaker to respect API limits

        statement = response_obj['statement']
        response_text = response_obj['response']
        prompt = JUDGE_TEMPLATE.format(STATEMENT=statement, RESPONSE=response_text)

        max_retries = 5
        mapped_choice = "UNKNOWN"

        for attempt in range(max_retries):
            try:
                api_response = await asyncio.wait_for(
                    client.aio.models.generate_content(
                        model=JUDGE_MODEL_ID,
                        contents=prompt,
                        config=judge_config
                    ),
                    timeout=20.0
                )

                if api_response.candidates and api_response.candidates[0].content.parts:
                    raw_choice = api_response.text.strip()
                    mapped_choice = map_to_abbreviation(raw_choice)

                    if mapped_choice in VALID_CHOICES:
                        break # Valid judgement secured
                    else:
                        await asyncio.sleep(1) # Retry if hallucinated

            except asyncio.TimeoutError:
                tqdm.write("⚠️ Network timeout. Retrying...")
                await asyncio.sleep(1)
            except Exception as e:
                error_msg = str(e).lower()
                if "429" in error_msg or "quota" in error_msg:
                    sleep_time = min(20, (1.5 ** attempt)) + random.uniform(0.1, 1.0)
                    tqdm.write(f"🚦 Quota hit. Sleeping {sleep_time:.1f}s...")
                    await asyncio.sleep(sleep_time)
                else:
                    tqdm.write(f"❌ Error judging: {error_msg}")
                    await asyncio.sleep(1)

        # Fallback to Neutral if all retries fail
        if mapped_choice not in VALID_CHOICES:
            tqdm.write(f"🚨 Failed to get valid judgement for a response. Defaulting to Neutral.")
            mapped_choice = "N"

        time_now = datetime.now(timezone.utc).isoformat()

        # Inject the judgement object
        response_obj["judgement"] = {
            "inference": mapped_choice,
            "Stance": STANCE_MAP[mapped_choice],
            "inference_value": VALUE_MAP[mapped_choice],
            "time": time_now
        }

        pbar.update(1)

async def run_judge_audit():
    print("\n--- STARTING MULTILINGUAL JUDGE PIPELINE ---")

    json_files = sorted([f for f in os.listdir(RESPONSES_DIR) if f.endswith('.json')])
    if not json_files:
        print("No JSON files found in the responses directory.")
        return

    semaphore = asyncio.Semaphore(10) # Safe concurrency for Gemini API
    batch_size = 50 # Checkpoint interval

    for filename in json_files:
        filepath = os.path.join(RESPONSES_DIR, filename)

        # Load the model's response file
        with open(filepath, 'r', encoding='utf-8') as f:
            model_data = json.load(f)

        print(f"\nEvaluating: {filename}")

        # Flatten ALL responses to process them easily, enabling the pbar to track skips
        all_responses = []
        for item in model_data:
            for resp in item.get("responses", []):
                all_responses.append(resp)

        total_judgements = len(all_responses)

        # Optional: Pre-check if completely done so it skips the file entirely
        already_done = sum(1 for r in all_responses if "judgement" in r)
        if already_done == total_judgements and total_judgements > 0:
            print(f"✓ {filename} is already 100% evaluated. Skipping file.")
            continue

        print(f"Queued {total_judgements} total responses for judgement tracking.")

        with tqdm(total=total_judgements, desc=f"Judging") as pbar:
            for i in range(0, total_judgements, batch_size):
                batch = all_responses[i:i+batch_size]

                coroutines = [process_judgement(resp, semaphore, pbar) for resp in batch]
                await asyncio.gather(*coroutines)

                # Checkpoint: Save progress back to the same file
                with open(filepath, 'w', encoding='utf-8') as f:
                    json.dump(model_data, f, indent=4, ensure_ascii=False)

        print(f"✓ Completed and finalized {filename}.")

    print("\n🎉 All models fully judged and updated!")

# Execute the pipeline
await run_judge_audit()

## LDS Calculation

In [ ]:
import os
import json
import joblib
import pandas as pd
from datetime import datetime, timezone
from tqdm.auto import tqdm

# --- Configuration & Paths ---
RESPONSES_DIR = '/Runs/LDS/responses'
MODELS_DIR = '/Models'
PREDICTIONS_DIR = '/Runs/LDS/predictions'
OUTPUT_JSON_PATH = os.path.join(PREDICTIONS_DIR, 'lds_predictions.json')

os.makedirs(PREDICTIONS_DIR, exist_ok=True)

# 1. Extract and Flatten the Judged Data
print("Extracting judged responses from JSON files...")
all_records = []

json_files = sorted([f for f in os.listdir(RESPONSES_DIR) if f.endswith(".json")])

for filename in tqdm(json_files, desc="Loading JSONs"):
    filepath = os.path.join(RESPONSES_DIR, filename)
    model_name = filename.replace(".json", "")

    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

        for item in data:
            var = item.get("variable")
            year = item.get("year")

            for resp in item.get("responses", []):
                lang = resp.get("language")
                judgement = resp.get("judgement", {})

                # Default to 0.5 (Neutral) if missing
                inf_val = judgement.get("inference_value", 0.5)

                all_records.append({
                    "model": model_name,
                    "year": int(year),
                    "variable": var,
                    "language": lang,
                    "inference_value": float(inf_val)
                })

df_raw = pd.DataFrame(all_records)
print(f"✓ Extracted {len(df_raw)} total statements across all languages and models.")

# 2. Pivot the Data for the Model
# Rows = (model, year, language), Columns = variables
df_pivot = df_raw.pivot_table(
    index=['model', 'year', 'language'],
    columns='variable',
    values='inference_value',
    aggfunc='first'
).reset_index()

# Bulletproof structural gap filling
df_pivot = df_pivot.fillna(0.5)

# 3. Load Scikit-Learn Pipelines & Predict
print("\nLoading pipelines and predicting ideologies...")
years = [2009, 2014, 2019]
prediction_results = []
time_now = datetime.now(timezone.utc).isoformat()

for year in tqdm(years, desc="Predicting by Year"):
    src_path = os.path.join(MODELS_DIR, f'ideology_model_{year}.pkl')

    if not os.path.exists(src_path):
        print(f"⚠️ Warning: Pipeline not found at {src_path}. Skipping year {year}.")
        continue

    pipeline = joblib.load(src_path)
    year_data = df_pivot[df_pivot['year'] == year].copy()

    if year_data.empty:
        continue

    # Safely extract expected features
    if hasattr(pipeline, 'feature_names_in_'):
        expected_features = pipeline.feature_names_in_
    elif hasattr(pipeline.named_steps['scaler'], 'feature_names_in_'):
        expected_features = pipeline.named_steps['scaler'].feature_names_in_
    else:
        raise ValueError(f"Could not extract feature names from the {year} model.")

    # Align columns and apply missing values catch
    X = year_data.reindex(columns=expected_features, fill_value=0.5)

    # Predict Coordinates
    predictions = pipeline.predict(X)

    # Append to results
    for i, idx in enumerate(year_data.index):
        prediction_results.append({
            'year': year,
            'model': year_data.loc[idx, 'model'],
            'language': year_data.loc[idx, 'language'],
            'lrgen': float(predictions[i][0]),
            'lrecon': float(predictions[i][1]),
            'galtan': float(predictions[i][2]),
            'prediction_time': time_now
        })

df_preds = pd.DataFrame(prediction_results)

# 4. Construct the Nested JSON Array
print("\nStructuring final JSON output...")
final_json_array = []

# Group by Year and Model
grouped = df_preds.groupby(['year', 'model'])

for (year, model), group in tqdm(grouped, desc="Nesting Data"):
    languages_array = []

    for _, row in group.iterrows():
        languages_array.append({
            "language": row['language'],
            "lrgen": row['lrgen'],
            "lrecon": row['lrecon'],
            "galtan": row['galtan'],
            "prediction_time": row['prediction_time']
        })

    final_json_array.append({
        "year": int(year),
        "model": model,
        "languages": languages_array
    })

# 5. Save to Drive
with open(OUTPUT_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(final_json_array, f, indent=4, ensure_ascii=False)

print("-" * 60)
print(f"✓ Success! Predicted ideologies saved structurally.")
print(f"✓ Total configurations mapped: {len(final_json_array)}")
print(f"✓ File saved to: {OUTPUT_JSON_PATH}")

In [ ]:
import os
import json
import pandas as pd
import numpy as np

# --- Configuration & Paths ---
INPUT_JSON_PATH = '/Runs/LDS/predictions/lds_predictions.json'
OUTPUT_CSV_PATH = '/Runs/LDS/lds.csv'

print("Loading predicted coordinates...")
with open(INPUT_JSON_PATH, 'r', encoding='utf-8') as f:
    data = json.load(f)

# 1. Flatten the Nested JSON into a DataFrame
records = []
for item in data:
    year = item['year']
    model = item['model']
    for lang_obj in item['languages']:
        records.append({
            'year': year,
            'model': model,
            'language': lang_obj['language'],
            'lrgen': lang_obj['lrgen'],
            'lrecon': lang_obj['lrecon'],
            'galtan': lang_obj['galtan']
        })

df = pd.DataFrame(records)

# 2. Isolate the English Baseline
df_eng = df[df['language'] == 'English'].copy()
df_eng = df_eng.rename(columns={
    'lrgen': 'lrgen_eng',
    'lrecon': 'lrecon_eng',
    'galtan': 'galtan_eng'
})
# Drop language column before merging so it doesn't collide
df_eng = df_eng.drop(columns=['language'])

# 3. Merge Baseline and Calculate Displacement
df_merged = pd.merge(df, df_eng, on=['year', 'model'], how='left')

# Calculate the 3D Euclidean displacement from the English anchor
df_merged['ches_displacement'] = np.sqrt(
    (df_merged['lrgen'] - df_merged['lrgen_eng'])**2 +
    (df_merged['lrecon'] - df_merged['lrecon_eng'])**2 +
    (df_merged['galtan'] - df_merged['galtan_eng'])**2
)

# 4. Aggregate LDS Summaries
# Group by model and language to get the average displacement across the years
lds_summary = df_merged.groupby(['model', 'language'])['ches_displacement'].mean().reset_index()

# Filter out the English-to-English comparisons (which are exactly 0)
lds_summary = lds_summary[lds_summary['language'] != 'English']

# Sort for readability: Model alphabetically, then highest displacement first
lds_summary = lds_summary.sort_values(by=['model', 'ches_displacement'], ascending=[True, False])

# 5. Save to Google Drive
lds_summary.to_csv(OUTPUT_CSV_PATH, index=False)

print("-" * 60)
print(f"✓ Success! LDS summary saved to: {OUTPUT_CSV_PATH}")
print("-" * 60)

# 6. Display Analytics in Colab
# Calculate the Global LDS per Model (Mean displacement across all foreign languages)
global_lds = lds_summary.groupby('model')['ches_displacement'].mean().reset_index()
global_lds = global_lds.sort_values(by='ches_displacement', ascending=False)

print("\n📊 [GLOBAL LDS BY MODEL]")
print("Higher score = Ideology is more sensitive to prompt language")
print(global_lds.to_string(index=False, float_format="%.4f"))

print("\n⚠️ [TOP 10 MOST EXTREME LINGUISTIC SHIFTS]")
display(lds_summary.nlargest(10, 'ches_displacement').style.format({'ches_displacement': '{:.4f}'}))